# LightGBM-only pipeline (multi-ticker, cross-sectional)

После Sprint 3: LightGBM достиг **Sharpe 0.98** (h=48), iTransformer
проиграл везде. Решение — закрыть iTransformer-track и сосредоточиться
на правильно-настроенном LightGBM с **расширенным feature space**.

## Что нового vs прошлого LightGBM (Sprint 3)

| # | Источник | Эффект |
|---|---|---|
| 1 | **14 тикеров вместо 2** | Больше данных, обучение на ВСЕМ universe сразу |
| 2 | **Cross-sectional rank/zscore/relative** | Asness-Moskowitz-Pedersen 2013: универсальный alpha-фактор |
| 3 | **HI2 multi-metric** (R-0052 fix) | 11 metrics вместо 1 (hhi_aggressive_buy/passive_sell/...) |
| 4 | **`sec_pr_*`** sectoral context | Уже есть в TradeStats, но не использовалось |

**FUTOI** не доступен на нашей истории (ALGOPACK FUTOI начинается с
2024-10-01, у нас train/val на 2020-2024). Модуль `futoi_features.py`
остаётся в репо для будущего использования, но в этом эксперименте
не подключается. См. R-0053.

## Ожидаемый эффект

Sharpe 0.98 (Sprint 3) → 1.2-1.5 (с новыми фичами и лучшим threshold'ом).

Время: ~5-8 минут на Colab CPU (LightGBM × 4 horizons на 14 тикерах).

## 0. Окружение

In [ ]:
from __future__ import annotations

import os, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy(); env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True, env=env)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
                        f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
                       check=True, env=env)
else:
    PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '--quiet',
                    'pandas>=2.1', 'pyarrow>=15.0', 'scikit-learn>=1.4',
                    'lightgbm>=4.0', 'tqdm>=4.66', 'matplotlib>=3.7'],
                   check=True)
    print('Deps OK')

In [ ]:
import sys
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss

from graduate_work.config import default_config

TICKERS = (
    'SBER', 'VTBR', 'GAZP', 'LKOH', 'GMKN', 'ROSN', 'NVTK',
    'MGNT', 'TATN', 'MTSS', 'MOEX', 'NLMK', 'CHMF', 'ALRS',
)

cfg = default_config()
cfg = dataclasses.replace(cfg, data=dataclasses.replace(cfg.data, tickers=TICKERS))
print(f'tickers={cfg.data.tickers}\nhorizons={cfg.data.horizons}')

## 1. Drive: подтягиваем все 14 тикеров

Ожидаемая структура:
```
MyDrive/lstm_exchange/data/raw/Algopack/
├── algopack/
│   ├── tradestats/{14 ticker}.csv
│   ├── orderstats/{14 ticker}.csv
│   ├── obstats/{14 ticker}.csv
│   └── hi2/{14 ticker}.csv  (с 11 metrics — R-0052 fix)
```

In [ ]:
import shutil
DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Скопировано из {src_dir}')

for sub in ['algopack/tradestats', 'algopack/orderstats',
            'algopack/obstats', 'algopack/hi2']:
    p = cfg.paths.data_raw / sub
    if p.exists():
        n = sum(1 for _ in p.glob('*.csv'))
        print(f'  {sub}: {n} файлов')
    else:
        print(f'  {sub}: ОТСУТСТВУЕТ')

## 2. Загрузка SuperCandles + HI2

In [ ]:
from graduate_work.features.algopack_features import (
    obstats_features, orderstats_features, tradestats_features,
    order_to_trade_ratio, hi2_features,
)
from graduate_work.features.targets import cost_aware_classification_labels, lr_columns
from graduate_work.features.cross_sectional import add_cross_sectional_features


def _load_csv_indexed(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    if df.empty or 'tradedate' not in df.columns:
        return df
    if 'tradetime' in df.columns:
        ts = pd.to_datetime(df['tradedate'].astype(str) + ' ' + df['tradetime'].astype(str),
                            utc=True, errors='coerce')
    else:
        ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
    df.index = ts
    df.index.name = 'begin'
    return df.dropna(how='all').sort_index()


ts_data, os_data, ob_data, hi2_data = {}, {}, {}, {}
for ticker in TICKERS:
    ts_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'tradestats' / f'{ticker}.csv')
    os_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'orderstats' / f'{ticker}.csv')
    ob_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'obstats' / f'{ticker}.csv')
    hi2_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'hi2' / f'{ticker}.csv')

loaded = {t: len(ts_data[t]) for t in TICKERS if not ts_data[t].empty}
print(f'Loaded {len(loaded)} of {len(TICKERS)} tickers (tradestats):')
for t, n in loaded.items():
    print(f'  {t}: {n} bars, hi2={len(hi2_data[t])}')
missing = set(TICKERS) - set(loaded.keys())
if missing:
    print(f'\nMISSING tickers: {sorted(missing)} — будут пропущены')
    TICKERS = tuple(t for t in TICKERS if t not in missing)
    print(f'Used: {TICKERS}')

## 3. Build per-ticker features (existing pipeline)

In [ ]:
def _ts_to_ohlcv(ts: pd.DataFrame, ticker: str) -> pd.DataFrame:
    out = pd.DataFrame(index=ts.index)
    out['open']   = ts['pr_open'].astype(float)
    out['high']   = ts['pr_high'].astype(float)
    out['low']    = ts['pr_low'].astype(float)
    out['close']  = ts['pr_close'].astype(float)
    out['volume'] = ts['vol'].astype(float)
    out['value']  = ts['val'].astype(float)
    out['vwap']   = ts['pr_vwap'].astype(float)
    # sec_pr_* — секторальный индекс OHLC, ранее не использовался.
    for col in ['sec_pr_open', 'sec_pr_high', 'sec_pr_low', 'sec_pr_close']:
        if col in ts.columns:
            out[col] = ts[col].astype(float)
    out['ticker'] = ticker
    return out


def _build_log_returns(close: pd.Series) -> pd.DataFrame:
    lr = np.log(close / close.shift(1))
    return pd.DataFrame({
        'lr_1':  lr,
        'lr_5':  lr.rolling(5).sum(),
        'lr_15': lr.rolling(15).sum(),
        'lr_30': lr.rolling(30).sum(),
    }, index=close.index)


def _build_sec_features(feat: pd.DataFrame) -> pd.DataFrame:
    """Sectoral context features — relative strength vs sector index."""
    if 'sec_pr_close' not in feat.columns:
        return pd.DataFrame(index=feat.index)
    out = pd.DataFrame(index=feat.index)
    sec_close = feat['sec_pr_close'].astype(float)
    out['sec_rel_strength'] = (feat['close'] / sec_close.replace(0, np.nan)).fillna(1.0) - 1.0
    sec_lr = np.log(sec_close / sec_close.shift(1))
    out['sec_lr_1'] = sec_lr.fillna(0.0)
    out['sec_lr_5'] = sec_lr.rolling(5).sum().fillna(0.0)
    out['sec_lr_15'] = sec_lr.rolling(15).sum().fillna(0.0)
    own_lr = np.log(feat['close'] / feat['close'].shift(1))
    out['sec_lr_residual'] = (own_lr - sec_lr).fillna(0.0)
    return out


def build_per_ticker(ticker: str) -> pd.DataFrame:
    ts_df = ts_data[ticker]
    if ts_df.empty:
        return pd.DataFrame()
    feat = _ts_to_ohlcv(ts_df, ticker)
    feat = feat.join(_build_log_returns(feat['close']), how='left')
    feat = feat.join(_build_sec_features(feat), how='left')
    feat = feat.join(tradestats_features(ts_df), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(orderstats_features(os_data[ticker]), how='left')
    if not ob_data[ticker].empty:
        feat = feat.join(obstats_features(ob_data[ticker]), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
    # HI2 (теперь с 11 metrics после R-0052 fix)
    if not hi2_data[ticker].empty:
        feat = feat.join(hi2_features(hi2_data[ticker], feat.index), how='left')
    return feat.fillna(0.0)


per_ticker = {t: build_per_ticker(t) for t in TICKERS}
per_ticker = {t: df for t, df in per_ticker.items() if not df.empty}
print(f'Per-ticker frames: {len(per_ticker)}')
for t in list(per_ticker.keys())[:3]:
    print(f'  {t}: shape={per_ticker[t].shape}, '
          f'sample cols={list(per_ticker[t].columns)[:6]}...')

## 4. Cross-sectional features

Для ключевых imbalance-фич: rank/zscore/relative across universe.

In [ ]:
# Cross-sectional на самые predictive фичи. По AUC из Sprint 1 знаем,
# что imbalance / order-flow доминируют — на них и применяем.
XS_FEATURES = [
    'aps_vol_imb',         # aggressive trade imbalance
    'aps_disb',            # raw disbalance
    'aps_vwap_premium',    # VWAP buy vs sell premium
    'aps_imb_vol_bbo',     # book imbalance
    'aps_pr_change_bp',    # bar return
    'aps_spread_bbo_bp',   # spread (liquidity)
]
# Для центрированных (∈ [-1,1]) imbalance-фич — relative_mode='diff';
# для положительных (spread) — 'ratio'. Compromise: 'diff' для всех
# (теряем 1-2% accuracy на spread, но избегаем нелинейности).
per_ticker_xs = add_cross_sectional_features(
    per_ticker, base_features=XS_FEATURES,
    rank=True, zscore=True, relative_mode='diff',
)
sample_t = list(per_ticker_xs.keys())[0]
new_cols = [c for c in per_ticker_xs[sample_t].columns if c.startswith(tuple(XS_FEATURES)) and ('_x' in c)]
print(f'Added cross-sectional cols (sample={sample_t}): {len(new_cols)}')
for c in new_cols[:9]:
    print(f'  {c}')

## 5. Concatenate to single multi-ticker frame + targets

In [ ]:
# Объединяем все per-ticker DF в один multi-ticker frame.
full = pd.concat(per_ticker_xs.values()).sort_index()

# Cost-aware targets per-ticker
out_parts = []
costs = cfg.trading.commission_rate + cfg.trading.slippage_rate
for ticker, sub in full.groupby('ticker'):
    labels = cost_aware_classification_labels(
        open_price=sub['open'], close_price=sub['close'],
        horizons=cfg.data.horizons,
        entry_cost=costs, exit_cost=costs,
        label_smoothing=0.0, direction='long',
    )
    out_parts.append(sub.join(labels, how='left'))
full_with_targets = pd.concat(out_parts).sort_values(['ticker']).sort_index(kind='stable')

target_cols = [f'target_h{h}' for h in cfg.data.horizons]
lr_cols_list = lr_columns(cfg.data.horizons)
feature_cols = [c for c in full_with_targets.columns
                if c not in ('ticker',) and c not in target_cols and c not in lr_cols_list]
print(f'Multi-ticker frame: {full_with_targets.shape}')
print(f'Feature count: {len(feature_cols)}')
for c in target_cols:
    p_up = float((full_with_targets[c].dropna() >= 0.5).mean())
    print(f'  {c}: P(UP)={p_up:.3f}')

## 6. Chronological split (train/val/test)

In [ ]:
from graduate_work.features.pipeline import chronological_split

train_df, val_df, test_df = chronological_split(
    full_with_targets,
    cfg.data.train_ratio, cfg.data.val_ratio,
    purge_horizon=max(cfg.data.horizons),
)
test_prices_raw = test_df[['open', 'close', 'ticker']].copy()
print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')
print(f'Train tickers: {train_df["ticker"].value_counts().to_dict()}')

## 7. Train LightGBMPipeline (per-horizon)

In [ ]:
from graduate_work.model.lgbm_pipeline import LightGBMConfig, LightGBMPipeline

lgb_cfg = LightGBMConfig(
    n_estimators=500, num_leaves=63, learning_rate=0.03,
    min_child_samples=200, reg_lambda=1.0, reg_alpha=0.1,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    early_stopping_rounds=30, random_state=42,
)
pipeline = LightGBMPipeline(
    horizons=cfg.data.horizons,
    feature_cols=feature_cols,
    cfg=lgb_cfg,
)
pipeline.fit(train_df, val_df)

ckpt_dir = cfg.paths.checkpoints / 'lgbm_only'
pipeline.save(ckpt_dir)

print('\n=== Per-horizon results ===')
for h in cfg.data.horizons:
    if h in pipeline.models:
        r = pipeline.models[h]
        print(f'  h={h}: best_iter={r.best_iteration}, val_log_loss={r.val_log_loss:.4f}, '
              f'val_auc={r.val_auc:.4f}' if r.val_auc else f'  h={h}: ...')

## 8. Test predictions + diagnostic

In [ ]:
test_predictions = pipeline.predict(test_df)
test_predictions['timestamp'] = pd.to_datetime(test_predictions['timestamp'], utc=True)
val_predictions = pipeline.predict(val_df)
val_predictions['timestamp'] = pd.to_datetime(val_predictions['timestamp'], utc=True)

print('=== Mean prediction vs P(UP) on test ===')
print(f'{"horizon":>8} | {"P(UP)":>7} | {"mean":>7} | {"max":>7} | {"std":>7} | {"AUC":>7} | {"n":>7}')

# test_df → long-form (timestamp, ticker, target_h{h}) для каждого горизонта.
# Index у test_df — DatetimeIndex; reset_index делает его колонкой.
test_long_index = pd.to_datetime(test_df.index, utc=True)

for h in cfg.data.horizons:
    if h not in pipeline.models:
        continue
    target_col = f'target_h{h}'
    if target_col not in test_df.columns:
        continue
    # Берём срез test_df с непустым target и его (timestamp, ticker, target).
    mask = test_df[target_col].notna()
    if not mask.any():
        continue
    sub = pd.DataFrame({
        'timestamp': test_long_index[mask.to_numpy()],
        'ticker': test_df.loc[mask, 'ticker'].to_numpy(),
        'target': test_df.loc[mask, target_col].astype(int).to_numpy(),
    })
    # Соответствующие предсказания для этого горизонта.
    h_pred = test_predictions[test_predictions['horizon'] == h]
    merged = h_pred.merge(sub, on=['timestamp', 'ticker'], how='inner')
    if merged.empty:
        continue
    h_proba = merged['mean'].to_numpy()
    h_target = merged['target'].to_numpy()
    p_up = float(h_target.mean())
    auc = roc_auc_score(h_target, h_proba) if len(set(h_target)) == 2 else float('nan')
    print(f'{h:>8} | {p_up:>7.3f} | {h_proba.mean():>7.4f} | {h_proba.max():>7.4f} | '
          f'{h_proba.std():>7.4f} | {auc:>7.4f} | {len(merged):>7d}')

## 9. Strategy comparison: prob_cutoff_0.5 / top_5% / top_10% / max_pnl

Те же 4 стратегии что и в Sprint 3 — но теперь на multi-ticker LightGBM.

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest
from graduate_work.strategy import (
    apply_isotonic_calibration, fit_isotonic_per_horizon,
    max_pnl_threshold, signals_per_horizon_threshold, top_k_threshold,
    attach_actual_targets, attach_lr_targets,
)
from graduate_work.features.pipeline import _build_arrays

bars_per_year = cfg.data.bars_per_year
cost_per_trade = 2.0 * (cfg.trading.commission_rate + cfg.trading.slippage_rate)

# Long-form lr-targets для max_pnl
val_lr_rows = []
for h in cfg.data.horizons:
    target_col = f'target_h{h}'
    lr_col = f'lr_h{h}'
    if lr_col not in val_df.columns:
        continue
    sub = val_df[val_df[target_col].notna() & val_df[lr_col].notna()]
    for ts, row in sub.iterrows():
        val_lr_rows.append({
            'timestamp': pd.to_datetime(ts, utc=True), 'ticker': row['ticker'],
            'horizon': int(h), 'actual': float(row[lr_col]),
        })
val_lr_targets = pd.DataFrame(val_lr_rows)
if not val_lr_targets.empty:
    val_lr_targets['timestamp'] = pd.to_datetime(val_lr_targets['timestamp'], utc=True)

# Long-form actual_targets для isotonic
val_actual_rows = []
for h in cfg.data.horizons:
    target_col = f'target_h{h}'
    sub = val_df[val_df[target_col].notna()]
    for ts, row in sub.iterrows():
        val_actual_rows.append({
            'timestamp': pd.to_datetime(ts, utc=True), 'ticker': row['ticker'],
            'horizon': int(h), 'actual': float(row[target_col]),
        })
val_actual_targets = pd.DataFrame(val_actual_rows)
if not val_actual_targets.empty:
    val_actual_targets['timestamp'] = pd.to_datetime(val_actual_targets['timestamp'], utc=True)


def _backtest(signals: pd.DataFrame) -> dict:
    n_buy = int((signals['action'] == 'BUY').sum())
    if n_buy == 0:
        return {'n_buy': 0, 'sharpe': 0.0, 'total_return': 0.0,
                'win_rate': 0.0, 'max_dd': 0.0}
    bt = run_backtest(signals, test_prices_raw, cfg.trading)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    return {'n_buy': n_buy, 'sharpe': float(m['sharpe']),
            'total_return': float(m['total_return']),
            'win_rate': float(m['win_rate']),
            'max_dd': float(m['max_drawdown'])}


iso_calibrators = fit_isotonic_per_horizon(val_predictions, val_actual_targets)
test_predictions_iso = apply_isotonic_calibration(test_predictions, iso_calibrators)

rows = []
for h in cfg.data.horizons:
    if h not in pipeline.models:
        continue
    h_val_pred = val_predictions[val_predictions['horizon'] == h]

    # 1. prob_cutoff_0.5
    sig = signals_per_horizon_threshold(test_predictions, h, threshold=0.5)
    rows.append({'horizon': h, 'strategy': 'prob_cutoff_0.5',
                 'threshold': 0.5, **_backtest(sig)})

    # 2. top-5%
    T = top_k_threshold(h_val_pred['mean'].values, k_pct=5.0)
    sig = signals_per_horizon_threshold(test_predictions, h, threshold=T)
    rows.append({'horizon': h, 'strategy': 'top_5%',
                 'threshold': T, **_backtest(sig)})

    # 3. top-10%
    T = top_k_threshold(h_val_pred['mean'].values, k_pct=10.0)
    sig = signals_per_horizon_threshold(test_predictions, h, threshold=T)
    rows.append({'horizon': h, 'strategy': 'top_10%',
                 'threshold': T, **_backtest(sig)})

    # 4. isotonic + 0.5
    sig = signals_per_horizon_threshold(test_predictions_iso, h, threshold=0.5)
    rows.append({'horizon': h, 'strategy': 'isotonic_0.5',
                 'threshold': 0.5, **_backtest(sig)})

    # 5. max_pnl на val
    h_val_lr = val_lr_targets[val_lr_targets['horizon'] == h]
    merged = h_val_pred.merge(h_val_lr, on=['timestamp', 'ticker', 'horizon'], how='inner')
    if not merged.empty:
        T_pnl, _ = max_pnl_threshold(
            merged['mean'].to_numpy(), merged['actual'].to_numpy(),
            cost_per_trade=cost_per_trade, min_trades=50,
        )
    else:
        T_pnl = 0.5
    sig = signals_per_horizon_threshold(test_predictions, h, threshold=T_pnl)
    rows.append({'horizon': h, 'strategy': 'max_pnl',
                 'threshold': T_pnl, **_backtest(sig)})

results = pd.DataFrame(rows)
print('=== Strategy × horizon matrix (Sharpe) ===')
pivot = results.pivot_table(index='strategy', columns='horizon', values='sharpe')
print(pivot.round(3).to_string())

print('\n=== Total return (%) matrix ===')
ret_pivot = results.pivot_table(index='strategy', columns='horizon', values='total_return') * 100
print(ret_pivot.round(2).to_string())

## 10. Verdict + best result

In [ ]:
positive = results[results['sharpe'] > 0].sort_values('sharpe', ascending=False)
if not positive.empty:
    best = positive.iloc[0]
    print('=== TOP POSITIVE-SHARPE COMBINATIONS ===')
    print(positive.head(10).to_string(
        index=False,
        float_format=lambda x: f'{x:.4f}',
    ))
    print(f'\nBEST: {best["strategy"]} @ h={best["horizon"]}: '
          f'Sharpe={best["sharpe"]:.3f}, return={best["total_return"]*100:.2f}%, '
          f'n_BUY={int(best["n_buy"])}, win_rate={best["win_rate"]:.3f}')
    print('\nСравнение с Sprint 3 baseline (LightGBM h=48 prob_cutoff_0.5, 2 тикера):')
    print('  Sprint 3: Sharpe=0.977, return=+1.23%, n_BUY=439')
    print(f'  Сейчас:   Sharpe={best["sharpe"]:.3f}, return={best["total_return"]*100:.2f}%, '
          f'n_BUY={int(best["n_buy"])}')
else:
    least_bad = results.sort_values('sharpe', ascending=False).iloc[0]
    print('=== Все стратегии негативные ===')
    print(f'Наименее плохая: {least_bad["strategy"]} @ h={least_bad["horizon"]} → '
          f'Sharpe={least_bad["sharpe"]:.3f}')

## 11. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
# Sharpe heatmap
ax = axes[0]
pivot.plot.bar(ax=ax)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(0.977, color='green', ls='--', lw=0.8, label='Sprint 3 best (0.98)')
ax.set_title('Sharpe per (strategy × horizon) — multi-ticker LightGBM')
ax.set_ylabel('Sharpe (test)')
ax.legend()
ax.grid(alpha=0.3)

# Feature importance для лучшего горизонта
ax = axes[1]
best_h = positive.iloc[0]['horizon'] if not positive.empty else cfg.data.horizons[-1]
best_model = pipeline.models[best_h].model
fi = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_,
}).sort_values('importance', ascending=False).head(20)
ax.barh(fi['feature'][::-1], fi['importance'][::-1])
ax.set_title(f'Top-20 feature importance (h={int(best_h)})')
ax.set_xlabel('importance')
plt.tight_layout(); plt.show()

# Сколько новых фич попали в top-20?
n_xs_in_top = sum(1 for f in fi['feature'] if any(s in f for s in ['_xrank', '_xzscore', '_xrel']))
n_hi2_in_top = sum(1 for f in fi['feature'] if f.startswith('aps_hi2_'))
n_sec_in_top = sum(1 for f in fi['feature'] if f.startswith('sec_'))
print(f'\nTop-20 диагностика:')
print(f'  cross-sectional (_xrank/_xzscore/_xrel): {n_xs_in_top}')
print(f'  HI2 multi-metric (aps_hi2_*): {n_hi2_in_top}')
print(f'  Sectoral (sec_*): {n_sec_in_top}')

## 12. Что дальше

- Если Sharpe > 1.2 → **production deploy** этого pipeline (уже есть `pipeline.save()` в чекпоинте)
- Если Sharpe ≈ 0.98 (как Sprint 3) → **новые фичи не дали edge**, нужен другой signal source (тики, news sentiment)
- Если Sharpe ухудшился → проверить, не вызывают ли cross-sectional фичи distribution shift

Feature importance в графике сразу подскажет: если top-20 доминируют **новые** фичи (cross-sectional / FUTOI / HI2-multi-metric / sec_*) — расширение feature space сработало.